# 🏙️ Xontaxtadagi mini shaharcha — 3D animatsiya

Xontaxta ustida kichkina shaharcha: **10 ta bino**, daraxtlar, yurib yurgan **odamchalar**, yo'lda yuradigan **mashinalar va avtobus**, charxpalak, favvora, osmonda aylanib yurgan **vertolyot** va stol chetida choynak, piyolalar va non.

**Qanday ishga tushiriladi:** pastdagi kod katagining chap tomonidagi ▶ tugmasini bosing (yoki menyudan *Runtime → Run all*). Bir necha soniyada animatsiya shu yerda chiqadi.

**Boshqarish:**
- 🖱️ Sichqoncha bilan tortib aylantiring, g'ildirak bilan yaqinlashtiring yoki uzoqlashtiring (telefonda barmoq bilan)
- 🌙 **Tun / ☀️ Kun** — shahar chiroqlari va derazalar yonadi
- ⏸ **Aylanish** — kameraning avtomatik aylanishini to'xtatadi yoki yoqadi
- ⏩ **Tezlik** — 1x → 2x → 0.5x
- 🎥 **10 soniya video yozish** — animatsiyani `.webm` video qilib yuklab beradi

In [ ]:
from IPython.display import HTML, display

html = r'''<div id="mc-wrap" style="position:relative;width:100%;height:620px;background:#dfe9f3;border-radius:12px;overflow:hidden;font-family:Arial,sans-serif">
  <div id="mc-view" style="position:absolute;inset:0"></div>
  <div style="position:absolute;left:12px;top:10px;color:#223;background:rgba(255,255,255,.75);padding:6px 10px;border-radius:8px;font-size:14px">
    <b>Xontaxtadagi mini shaharcha</b> · sichqoncha bilan aylantiring, g'ildirak bilan yaqinlashtiring
  </div>
  <div style="position:absolute;left:12px;bottom:12px;display:flex;gap:8px;flex-wrap:wrap">
    <button id="mc-night" class="mc-btn">🌙 Tun</button>
    <button id="mc-rotate" class="mc-btn">⏸ Aylanishni to'xtatish</button>
    <button id="mc-speed" class="mc-btn">⏩ Tezlik: 1x</button>
    <button id="mc-rec" class="mc-btn">🎥 10 soniya video yozish</button>
  </div>
  <div id="mc-msg" style="position:absolute;right:12px;bottom:12px;color:#fff;background:rgba(0,0,0,.55);padding:6px 10px;border-radius:8px;font-size:13px;display:none"></div>
</div>
<style>
.mc-btn{border:0;border-radius:8px;padding:8px 12px;background:rgba(255,255,255,.9);color:#123;font-size:14px;cursor:pointer;box-shadow:0 1px 4px rgba(0,0,0,.25)}
.mc-btn:hover{background:#fff}
</style>
<script src="https://cdn.jsdelivr.net/npm/three@0.147.0/build/three.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/three@0.147.0/examples/js/controls/OrbitControls.js"></script>
<script>
(function(){
const view = document.getElementById('mc-view');
const W = () => view.clientWidth, H = () => view.clientHeight;

// ---------- Renderer, sahna, kamera ----------
const renderer = new THREE.WebGLRenderer({antialias:true, preserveDrawingBuffer:true});
renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
renderer.setSize(W(), H());
renderer.shadowMap.enabled = true;
renderer.shadowMap.type = THREE.PCFSoftShadowMap;
view.appendChild(renderer.domElement);

const scene = new THREE.Scene();
const DAY_BG = new THREE.Color(0xdfe9f3), NIGHT_BG = new THREE.Color(0x0b1026);
scene.background = DAY_BG.clone();
scene.fog = new THREE.Fog(scene.background, 40, 90);

const camera = new THREE.PerspectiveCamera(45, W()/H(), 0.1, 200);
camera.position.set(13, 12, 15);
if (W()/H() < 1) camera.position.set(22, 20, 26);   // telefon (tik ekran) uchun uzoqroqdan
const controls = new THREE.OrbitControls(camera, renderer.domElement);
controls.target.set(0, 2.2, 0);
controls.enableDamping = true;
controls.autoRotate = true;
controls.autoRotateSpeed = 0.6;
controls.maxPolarAngle = Math.PI * 0.48;
controls.minDistance = 4; controls.maxDistance = 45;

// ---------- Yorug'lik ----------
const hemi = new THREE.HemisphereLight(0xffffff, 0x8a7a66, 0.75);
scene.add(hemi);
const sun = new THREE.DirectionalLight(0xfff2dd, 0.95);
sun.position.set(10, 20, 8);
sun.castShadow = true;
sun.shadow.mapSize.set(2048, 2048);
Object.assign(sun.shadow.camera, {left:-14, right:14, top:14, bottom:-14, near:1, far:60});
sun.shadow.bias = -0.0005;
scene.add(sun);
const moonLight = new THREE.PointLight(0xffd28a, 0, 16);   // tunda shahar ustidagi iliq nur
moonLight.position.set(-1.5, 6, 0);
scene.add(moonLight);

// ---------- Yordamchi funksiyalar ----------
function canvasTex(w, h, draw){
  const c = document.createElement('canvas'); c.width = w; c.height = h;
  draw(c.getContext('2d'), w, h);
  const t = new THREE.CanvasTexture(c);
  return t;
}
const std = (color, extra={}) => new THREE.MeshStandardMaterial(Object.assign({color, roughness:0.8}, extra));
function mesh(geo, mat, x=0, y=0, z=0, parent=scene){
  const m = new THREE.Mesh(geo, mat);
  m.position.set(x, y, z); m.castShadow = true; m.receiveShadow = true;
  parent.add(m); return m;
}
const rand = (a, b) => a + Math.random() * (b - a);
const pick = arr => arr[Math.floor(Math.random() * arr.length)];

// ---------- Xona: pol va gilam ----------
const floorTex = canvasTex(512, 512, (g, w, h) => {
  g.fillStyle = '#5a4230'; g.fillRect(0, 0, w, h);
  for (let i = 0; i < 16; i++){ g.fillStyle = i % 2 ? '#654a35' : '#6e5139'; g.fillRect(0, i*32, w, 30); }
});
floorTex.wrapS = floorTex.wrapT = THREE.RepeatWrapping; floorTex.repeat.set(6, 6);
const floor = mesh(new THREE.PlaneGeometry(120, 120), std(0xffffff, {map:floorTex}));
floor.rotation.x = -Math.PI/2; floor.castShadow = false;

const rugTex = canvasTex(1024, 768, (g, w, h) => {
  g.fillStyle = '#8e1b1b'; g.fillRect(0, 0, w, h);
  g.strokeStyle = '#e8c07a'; g.lineWidth = 18; g.strokeRect(30, 30, w-60, h-60);
  g.strokeStyle = '#1d2f5e'; g.lineWidth = 10; g.strokeRect(62, 62, w-124, h-124);
  for (let x = 140; x < w-100; x += 120) for (let y = 140; y < h-100; y += 120){
    g.fillStyle = (x + y) % 240 ? '#e8c07a' : '#1d2f5e';
    g.beginPath(); g.moveTo(x, y-40); g.lineTo(x+40, y); g.lineTo(x, y+40); g.lineTo(x-40, y); g.closePath(); g.fill();
    g.fillStyle = '#8e1b1b'; g.beginPath(); g.arc(x, y, 12, 0, 7); g.fill();
  }
});
const rug = mesh(new THREE.PlaneGeometry(28, 21), std(0xffffff, {map:rugTex, roughness:1}), 0, 0.02, 0);
rug.rotation.x = -Math.PI/2; rug.castShadow = false;

// ---------- Xontaxta ----------
const woodTex = canvasTex(512, 256, (g, w, h) => {
  g.fillStyle = '#7a4a24'; g.fillRect(0, 0, w, h);
  for (let i = 0; i < 60; i++){
    g.strokeStyle = `rgba(${rand(40,70)|0},${rand(20,35)|0},10,${rand(.15,.4)})`;
    g.lineWidth = rand(1, 4); g.beginPath();
    const y = rand(0, h); g.moveTo(0, y);
    g.bezierCurveTo(w*.3, y + rand(-10,10), w*.6, y + rand(-10,10), w, y + rand(-6,6)); g.stroke();
  }
});
const wood = std(0xffffff, {map:woodTex, roughness:0.6});
const TABLE_Y = 2.0;                       // stol usti balandligi
mesh(new THREE.BoxGeometry(17, 0.35, 11), wood, 0, TABLE_Y - 0.175, 0);
mesh(new THREE.BoxGeometry(16.2, 0.5, 10.2), wood, 0, TABLE_Y - 0.6, 0);   // yon taxtalar
for (const sx of [-1, 1]) for (const sz of [-1, 1]){
  mesh(new THREE.BoxGeometry(0.7, TABLE_Y - 0.35, 0.7), wood, sx*7.7, (TABLE_Y - 0.35)/2, sz*4.7);
}

// ---------- Shahar asosi ----------
const city = new THREE.Group();
city.position.set(-1.6, TABLE_Y, 0);        // o'ng tomonda choynak uchun joy qoladi
scene.add(city);
mesh(new THREE.BoxGeometry(12.4, 0.08, 8.4), std(0x7fae5a), 0, 0.04, 0, city);   // o't-maysa

const A = 4.2, B = 2.3, R = 0.8, ROAD_W = 0.9;   // halqa yo'l o'lchamlari
const Y0 = 0.08;                                // shahar yer sathi

// Yo'l halqasini chizish uchun shakl (tashqi va ichki yumaloq to'rtburchak)
function roundRectShape(a, b, r){
  const s = new THREE.Shape();
  s.moveTo(-a + r, -b); s.lineTo(a - r, -b); s.quadraticCurveTo(a, -b, a, -b + r);
  s.lineTo(a, b - r); s.quadraticCurveTo(a, b, a - r, b); s.lineTo(-a + r, b);
  s.quadraticCurveTo(-a, b, -a, b - r); s.lineTo(-a, -b + r); s.quadraticCurveTo(-a, -b, -a + r, -b);
  return s;
}
function ring(a, b, r, w, mat, y){
  const outer = roundRectShape(a + w/2, b + w/2, r + w/2);
  outer.holes.push(roundRectShape(a - w/2, b - w/2, Math.max(r - w/2, 0.05)));
  const m = mesh(new THREE.ShapeGeometry(outer, 24), mat, 0, y, 0, city);
  m.rotation.x = -Math.PI/2; m.castShadow = false; return m;
}
const asphalt = std(0x3b3f45, {roughness:0.95});
const sidewalkMat = std(0xc9c3b5);
ring(A, B, R, ROAD_W + 0.7, sidewalkMat, Y0 + 0.005);   // trotuarlar
ring(A, B, R, ROAD_W, asphalt, Y0 + 0.012);             // yo'l
ring(A, B, R, 0.04, std(0xffffff, {emissive:0x333333}), Y0 + 0.016);   // o'rta chiziq
// O'rtadagi ko'ndalang yo'l va piyodalar o'tish joylari (zebra)
const mid = mesh(new THREE.PlaneGeometry(0.8, 2*B - ROAD_W), asphalt, 0, Y0 + 0.012, 0, city);
mid.rotation.x = -Math.PI/2; mid.castShadow = false;
for (const zz of [-1.2, 1.2]) for (let i = -3; i <= 3; i++){
  const z = mesh(new THREE.PlaneGeometry(0.07, 0.35), std(0xffffff), i*0.1, Y0 + 0.016, zz, city);
  z.rotation.x = -Math.PI/2; z.castShadow = false;
}

// Halqa yo'l bo'ylab harakat trayektoriyasi: s -> (x, z, yo'nalish)
function makeLoop(d){
  const segs = [], L1 = 2*(A - R), L2 = 2*(B - R), arc = Math.PI/2 * (R + d);
  segs.push({t:'l', len:L1, p:[-(A-R),  B+d], v:[ 1, 0]});
  segs.push({t:'a', len:arc, c:[ A-R,  B-R], th:Math.PI/2});
  segs.push({t:'l', len:L2, p:[ A+d,  B-R], v:[ 0,-1]});
  segs.push({t:'a', len:arc, c:[ A-R, -(B-R)], th:0});
  segs.push({t:'l', len:L1, p:[ A-R, -(B+d)], v:[-1, 0]});
  segs.push({t:'a', len:arc, c:[-(A-R), -(B-R)], th:-Math.PI/2});
  segs.push({t:'l', len:L2, p:[-(A+d), -(B-R)], v:[ 0, 1]});
  segs.push({t:'a', len:arc, c:[-(A-R),  B-R], th:Math.PI});
  const total = segs.reduce((s, g) => s + g.len, 0);
  return {total, at(s){
    s = ((s % total) + total) % total;
    for (const g of segs){
      if (s <= g.len){
        if (g.t === 'l') return {x:g.p[0] + g.v[0]*s, z:g.p[1] + g.v[1]*s, dx:g.v[0], dz:g.v[1]};
        const th = g.th - s/(R + d);
        return {x:g.c[0] + (R+d)*Math.cos(th), z:g.c[1] + (R+d)*Math.sin(th), dx:Math.sin(th), dz:-Math.cos(th)};
      }
      s -= g.len;
    }
  }};
}

// ---------- Binolar (10 ta) ----------
const winLit = [];   // tunda yonadigan deraza materiallari
function buildingTex(wall, cols, rows, lit){
  return canvasTex(256, 512, (g, w, h) => {
    g.fillStyle = lit ? '#000' : wall; g.fillRect(0, 0, w, h);
    const cw = w/cols, rh = h/rows;
    for (let i = 0; i < cols; i++) for (let j = 0; j < rows; j++){
      if (lit){ g.fillStyle = Math.random() < 0.6 ? pick(['#ffd27a','#ffe9b0','#fff3cf','#ffc05c']) : '#000'; }
      else g.fillStyle = Math.random() < 0.15 ? '#9fb6c9' : '#6f8da8';
      g.fillRect(i*cw + cw*0.2, j*rh + rh*0.22, cw*0.6, rh*0.56);
    }
  });
}
function building(x, z, w, d, h, wall, roofColor, roofType){
  const cols = Math.max(2, Math.round(w*4)), rows = Math.max(2, Math.round(h*4));
  const mat = std(0xffffff, {map:buildingTex(wall, cols, rows, false), emissiveMap:buildingTex(wall, cols, rows, true),
                             emissive:0xffffff, emissiveIntensity:0, roughness:0.7});
  winLit.push(mat);
  const roof = std(roofColor);
  const b = mesh(new THREE.BoxGeometry(w, h, d), [mat, mat, roof, roof, mat, mat], x, Y0 + h/2, z, city);
  if (roofType === 'pyramid'){
    const p = mesh(new THREE.ConeGeometry(Math.max(w, d)*0.72, 0.6, 4), roof, x, Y0 + h + 0.3, z, city);
    p.rotation.y = Math.PI/4;
  } else if (roofType === 'dome'){
    mesh(new THREE.SphereGeometry(Math.min(w, d)*0.42, 24, 12, 0, Math.PI*2, 0, Math.PI/2), std(0x2a9fd6, {metalness:.3, roughness:.4}), x, Y0 + h, z, city);
  } else {
    mesh(new THREE.BoxGeometry(w*0.35, 0.25, d*0.35), std(0x888888), x + w*0.15, Y0 + h + 0.125, z - d*0.1, city);   // tomdagi texnik xona
  }
  return b;
}
const B_LIST = [
  // x,    z,    w,   d,   h,   devor,     tom,      tom turi
  [-4.2,  3.55, 1.3, 0.8, 1.6, '#e8d8b8', 0xa0522d, 'pyramid'],
  [-1.9,  3.55, 1.6, 0.8, 2.6, '#cfd8dc', 0x546e7a, 'flat'],
  [ 0.6,  3.55, 1.2, 0.8, 1.2, '#f4c7a1', 0xc0392b, 'pyramid'],
  [ 3.2,  3.55, 1.8, 0.8, 2.0, '#fff4e0', 0x2a9fd6, 'dome'],
  [-3.6, -3.55, 1.8, 0.8, 3.4, '#b0bec5', 0x37474f, 'flat'],
  [-0.8, -3.55, 1.4, 0.8, 1.8, '#ffe0b2', 0x8d6e63, 'pyramid'],
  [ 2.6, -3.55, 2.0, 0.8, 4.2, '#90a4ae', 0x263238, 'flat'],
  [ 5.55, 0.0,  0.9, 2.2, 2.4, '#d7ccc8', 0x6d4c41, 'flat'],
  [ 1.65, 0.55, 1.2, 1.0, 3.0, '#e1f5fe', 0x0277bd, 'flat'],
  [ 2.75,-0.6,  1.0, 1.0, 1.9, '#fce4ec', 0xad1457, 'pyramid'],
];
B_LIST.forEach(b => building(...b));

// ---------- Daraxtlar ----------
const trees = [];
function tree(x, z, s=1){
  const g = new THREE.Group(); g.position.set(x, Y0, z); city.add(g);
  mesh(new THREE.CylinderGeometry(0.04*s, 0.06*s, 0.35*s, 6), std(0x6d4c2f), 0, 0.175*s, 0, g);
  const leafMat = std(pick([0x2e7d32, 0x388e3c, 0x43a047, 0x558b2f]));
  if (Math.random() < 0.5) mesh(new THREE.ConeGeometry(0.22*s, 0.6*s, 8), leafMat, 0, 0.6*s, 0, g);
  else { mesh(new THREE.SphereGeometry(0.22*s, 10, 8), leafMat, 0, 0.55*s, 0, g);
         mesh(new THREE.SphereGeometry(0.15*s, 10, 8), leafMat, 0.08*s, 0.72*s, 0.05*s, g); }
  g.userData.phase = Math.random()*6; trees.push(g);
}
[[-3.2,1.2],[-3.1,-1.2],[-1.0,-1.25],[-2.1,1.3],[-1.1,1.3],[-2.2,-1.3],
 [0.95,-1.3],[3.2,1.3],[1.0,1.35],[3.3,-1.35],
 [-5.7,3.6],[-5.7,-3.6],[5.6,3.6],[5.6,-3.6],[4.6,3.6],[-5.6,1.8],[-5.6,-1.8],[5.6,-2.0],[5.6,2.0],[0.9,-3.6]]
 .forEach(([x,z]) => tree(x, z, rand(0.85, 1.25)));

// ---------- Bog': charxpalak va favvora ----------
const wheelGroup = new THREE.Group(); wheelGroup.position.set(-2.55, Y0 + 1.05, 0); city.add(wheelGroup);
const metal = std(0xd0d4da, {metalness:.6, roughness:.35});
for (const zz of [-0.18, 0.18]) for (const sx of [-1, 1]){
  const leg = mesh(new THREE.CylinderGeometry(0.025, 0.025, 1.15, 6), metal, sx*0.28, -0.52, zz, wheelGroup);
  leg.rotation.z = sx*0.27;
}
const wheel = new THREE.Group(); wheelGroup.add(wheel);
const rim = mesh(new THREE.TorusGeometry(0.85, 0.025, 8, 48), std(0xe53935, {metalness:.3}), 0, 0, 0, wheel);
const cabins = [];
for (let i = 0; i < 10; i++){
  const a = i/10 * Math.PI*2;
  const spoke = mesh(new THREE.CylinderGeometry(0.012, 0.012, 0.85, 4), metal, Math.cos(a)*0.425, Math.sin(a)*0.425, 0, wheel);
  spoke.rotation.z = a + Math.PI/2;
  const cab = mesh(new THREE.BoxGeometry(0.16, 0.14, 0.16), std(pick([0xffb300, 0x29b6f6, 0x66bb6a, 0xec407a, 0xab47bc])), Math.cos(a)*0.85, Math.sin(a)*0.85 - 0.1, 0, wheel);
  wheel.remove(cab); wheelGroup.add(cab); cabins.push(cab);
}
// Favvora
const fx = -1.3, fz = 0.0;
mesh(new THREE.CylinderGeometry(0.45, 0.5, 0.12, 24), std(0xbdbdbd), fx, Y0 + 0.06, fz, city);
const water = mesh(new THREE.CylinderGeometry(0.4, 0.4, 0.02, 24), std(0x4fc3f7, {transparent:true, opacity:.85, roughness:.1}), fx, Y0 + 0.12, fz, city);
mesh(new THREE.CylinderGeometry(0.05, 0.07, 0.35, 10), std(0xbdbdbd), fx, Y0 + 0.2, fz, city);
const drops = [];
const dropMat = std(0x81d4fa, {transparent:true, opacity:.9, emissive:0x113355});
for (let i = 0; i < 40; i++){
  const d = mesh(new THREE.SphereGeometry(0.018, 6, 4), dropMat, fx, Y0 + 0.37, fz, city);
  d.castShadow = false; d.userData = {t:Math.random(), a:Math.random()*Math.PI*2, v:rand(.18, .3)}; drops.push(d);
}
// Skameykalar
for (const [x, z, r] of [[-1.6, 1.0, 0], [-1.6, -1.0, 0]]){
  const bench = mesh(new THREE.BoxGeometry(0.5, 0.04, 0.14), std(0x8d6e63), x, Y0 + 0.12, z, city); bench.rotation.y = r;
  mesh(new THREE.BoxGeometry(0.5, 0.12, 0.03), std(0x8d6e63), x, Y0 + 0.2, z + (z > 0 ? 0.06 : -0.06), city);
}

// ---------- Ko'cha chiroqlari ----------
const bulbs = [];
const bulbMat = std(0xfff3c4, {emissive:0xffd27a, emissiveIntensity:0});
for (const [x, z] of [[-3.4,3.0],[0,3.0],[3.4,3.0],[-3.4,-3.0],[0,-3.0],[3.4,-3.0],[4.95,1.2],[4.95,-1.2],[-4.95,1.2],[-4.95,-1.2]]){
  mesh(new THREE.CylinderGeometry(0.02, 0.025, 0.7, 6), std(0x37474f), x, Y0 + 0.35, z, city);
  bulbs.push(mesh(new THREE.SphereGeometry(0.06, 10, 8), bulbMat, x, Y0 + 0.72, z, city));
}

// ---------- Mashinalar ----------
const cars = [];
function car(color, lane, s0, speed, type){
  const g = new THREE.Group(); city.add(g);
  const len = type === 'bus' ? 0.9 : 0.5, hgt = type === 'bus' ? 0.26 : 0.13;
  mesh(new THREE.BoxGeometry(len, hgt, 0.26), std(color, {metalness:.35, roughness:.4}), 0, 0.1 + hgt/2, 0, g);
  if (type === 'bus'){
    mesh(new THREE.BoxGeometry(len*0.92, 0.08, 0.265), std(0x263238, {roughness:.2}), 0, 0.28, 0, g);   // derazalar qatori
  } else {
    mesh(new THREE.BoxGeometry(len*0.55, 0.11, 0.22), std(0x263238, {roughness:.2, metalness:.5}), -0.03, 0.28, 0, g);
    if (type === 'taxi') mesh(new THREE.BoxGeometry(0.1, 0.04, 0.06), std(0x111111), -0.03, 0.355, 0, g);
  }
  const head = std(0xfffde7, {emissive:0xfff3b0, emissiveIntensity:0});
  for (const zz of [-0.08, 0.08]){
    mesh(new THREE.BoxGeometry(0.02, 0.04, 0.05), head, len/2, 0.16, zz, g);
    mesh(new THREE.BoxGeometry(0.02, 0.04, 0.05), std(0xd50000, {emissive:0x550000}), -len/2, 0.16, zz, g);
  }
  const wheels = [];
  for (const wx of [-len*0.32, len*0.32]) for (const wz of [-0.13, 0.13]){
    const w = mesh(new THREE.CylinderGeometry(0.06, 0.06, 0.04, 12), std(0x111111), wx, 0.06, wz, g);
    w.rotation.x = Math.PI/2; wheels.push(w);
  }
  cars.push({g, loop:makeLoop(lane), s:s0, speed, wheels, head});
}
const carColors = [0xe53935, 0x1e88e5, 0xfdd835, 0x43a047, 0xffffff, 0x8e24aa, 0xfb8c00, 0x212121];
for (let i = 0; i < 5; i++) car(carColors[i], 0.22, i*5.2, 1.3, i === 2 ? 'taxi' : 'car');            // tashqi yo'lak
for (let i = 0; i < 3; i++) car(carColors[5 + i], -0.22, i*7.5 + 2, -1.05, 'car');                    // ichki yo'lak (teskari)
car(0x00897b, 0.22, 13.5, 1.3, 'bus');

// ---------- Odamchalar ----------
const people = [];
const skin = [0xf1c27d, 0xe0ac69, 0xc68642, 0xffdbac];
function person(lane, s0, speed){
  const g = new THREE.Group(); city.add(g);
  const shirt = std(pick([0xe53935, 0x1e88e5, 0xfdd835, 0x43a047, 0xab47bc, 0xff7043, 0x26c6da, 0xffffff]));
  const pants = std(pick([0x263238, 0x3949ab, 0x5d4037, 0x424242]));
  const body = mesh(new THREE.CylinderGeometry(0.045, 0.055, 0.16, 8), shirt, 0, 0.24, 0, g);
  mesh(new THREE.SphereGeometry(0.045, 10, 8), std(pick(skin)), 0, 0.36, 0, g);
  mesh(new THREE.SphereGeometry(0.047, 10, 6, 0, Math.PI*2, 0, Math.PI/2.2), std(pick([0x3e2723, 0x212121, 0x795548])), 0, 0.37, 0, g);   // soch
  const legs = [];
  for (const zz of [-0.025, 0.025]){
    const pivot = new THREE.Group(); pivot.position.set(0, 0.16, zz); g.add(pivot);
    mesh(new THREE.BoxGeometry(0.03, 0.16, 0.03), pants, 0, -0.08, 0, pivot);
    legs.push(pivot);
  }
  const arms = [];
  for (const zz of [-0.06, 0.06]){
    const pivot = new THREE.Group(); pivot.position.set(0, 0.3, zz); g.add(pivot);
    mesh(new THREE.BoxGeometry(0.025, 0.12, 0.025), shirt, 0, -0.06, 0, pivot);
    arms.push(pivot);
  }
  people.push({g, loop:makeLoop(lane), s:s0, speed, legs, arms, phase:Math.random()*6});
}
for (let i = 0; i < 9; i++)  person(0.62 + rand(-0.05, 0.05), i*3.1 + rand(0, 1), rand(0.22, 0.35) * (i % 3 ? 1 : -1));    // tashqi trotuar
for (let i = 0; i < 7; i++)  person(-0.62 + rand(-0.05, 0.05), i*3.3 + rand(0, 1), rand(0.2, 0.32) * (i % 2 ? 1 : -1));   // ichki trotuar

// Favvora atrofida aylanib yuradigan bolakaylar
const kids = [];
for (let i = 0; i < 3; i++){
  person(0, 0, 0); const p = people.pop();
  p.g.scale.setScalar(0.75); p.angle = i * 2.1; kids.push(p);
}

// ---------- Choynak, piyolalar va non ----------
const tea = new THREE.Group(); tea.position.set(6.3, TABLE_Y, -1.2); scene.add(tea);
const porcelain = std(0xf5f7ff, {roughness:.25});
const cobalt = std(0x1a3fa8, {roughness:.3});
mesh(new THREE.SphereGeometry(0.75, 32, 24), porcelain, 0, 0.72, 0, tea).scale.y = 0.95;
mesh(new THREE.TorusGeometry(0.62, 0.05, 8, 40), cobalt, 0, 0.9, 0, tea).rotation.x = Math.PI/2;
mesh(new THREE.TorusGeometry(0.7, 0.04, 8, 40), cobalt, 0, 0.55, 0, tea).rotation.x = Math.PI/2;
mesh(new THREE.CylinderGeometry(0.28, 0.32, 0.12, 24), porcelain, 0, 1.46, 0, tea);
mesh(new THREE.SphereGeometry(0.1, 12, 8), cobalt, 0, 1.58, 0, tea);
const spout = mesh(new THREE.CylinderGeometry(0.06, 0.12, 0.8, 12), porcelain, -0.85, 0.95, 0, tea); spout.rotation.z = 0.9;
const handle = mesh(new THREE.TorusGeometry(0.33, 0.06, 8, 24, Math.PI*1.2), porcelain, 0.72, 0.85, 0, tea); handle.rotation.z = -Math.PI*0.6;
mesh(new THREE.CylinderGeometry(0.5, 0.45, 0.06, 24), porcelain, 0, 0.03, 0, tea);
for (const [x, z] of [[-0.1, 1.9], [1.0, 1.3]]){
  const cup = new THREE.Group(); cup.position.set(x, 0, z); tea.add(cup);
  mesh(new THREE.CylinderGeometry(0.34, 0.2, 0.32, 24, 1, true), porcelain, 0, 0.2, 0, cup).material.side = THREE.DoubleSide;
  mesh(new THREE.CylinderGeometry(0.2, 0.2, 0.04, 24), porcelain, 0, 0.04, 0, cup);
  mesh(new THREE.TorusGeometry(0.33, 0.02, 6, 32), cobalt, 0, 0.35, 0, cup).rotation.x = Math.PI/2;
  mesh(new THREE.CylinderGeometry(0.3, 0.3, 0.02, 24), std(0x8d5524, {roughness:.2}), 0, 0.28, 0, cup);   // choy
}
// Non
const nonTex = canvasTex(256, 256, (g, w, h) => {
  const gr = g.createRadialGradient(128, 128, 20, 128, 128, 128);
  gr.addColorStop(0, '#f3d9a4'); gr.addColorStop(0.45, '#e9c07a'); gr.addColorStop(0.55, '#b86b2b'); gr.addColorStop(1, '#a0561f');
  g.fillStyle = gr; g.fillRect(0, 0, w, h);
  g.fillStyle = '#8a4a1a';
  for (let i = 0; i < 40; i++){ const a = rand(0, 7), r = rand(0, 45); g.beginPath(); g.arc(128 + Math.cos(a)*r, 128 + Math.sin(a)*r, 3, 0, 7); g.fill(); }
});
const nonMesh = mesh(new THREE.CylinderGeometry(1.05, 1.1, 0.18, 40), [std(0xa0561f), std(0xffffff, {map:nonTex}), std(0xa0561f)], 6.3, TABLE_Y + 0.09, 2.6);
// Choynakdan ko'tarilayotgan bug'
const steam = [];
for (let i = 0; i < 14; i++){
  const s = mesh(new THREE.SphereGeometry(0.09, 8, 6), new THREE.MeshBasicMaterial({color:0xffffff, transparent:true, opacity:0}), 0, 0, 0, tea);
  s.castShadow = false; s.userData.t = i/14; steam.push(s);
}

// ---------- Vertolyot (shahar ustida aylanadi) ----------
const heli = new THREE.Group(); scene.add(heli);
mesh(new THREE.SphereGeometry(0.22, 16, 12), std(0xff7043, {metalness:.3}), 0, 0, 0, heli).scale.set(1.5, 1, 1);
mesh(new THREE.BoxGeometry(0.6, 0.06, 0.06), std(0xff7043), -0.5, 0.05, 0, heli);
mesh(new THREE.BoxGeometry(0.04, 0.2, 0.02), std(0x333333), -0.8, 0.12, 0, heli);
const rotor = mesh(new THREE.BoxGeometry(1.2, 0.015, 0.06), std(0x222222), 0, 0.26, 0, heli);
mesh(new THREE.CylinderGeometry(0.02, 0.02, 0.08, 6), std(0x333333), 0, 0.22, 0, heli);
const heliLight = mesh(new THREE.SphereGeometry(0.03, 6, 4), std(0xff1744, {emissive:0xff1744, emissiveIntensity:0}), 0, -0.2, 0, heli);

// ---------- Kun / tun ----------
let night = 0, nightTarget = 0;
function applyNight(n){
  scene.background.copy(DAY_BG).lerp(NIGHT_BG, n); scene.fog.color.copy(scene.background);
  hemi.intensity = 0.75 - 0.55*n;
  sun.intensity = 0.95 - 0.8*n;
  sun.color.setHex(n > 0.5 ? 0x8fa8ff : 0xfff2dd);
  moonLight.intensity = 0.9*n;
  winLit.forEach(m => m.emissiveIntensity = 1.1*n);
  bulbMat.emissiveIntensity = 2.5*n;
  cars.forEach(c => c.head.emissiveIntensity = 2*n);
  heliLight.material.emissiveIntensity = 3*n;
}

// ---------- Tugmalar ----------
let speedMul = 1;
const $ = id => document.getElementById(id);
$('mc-night').onclick = () => { nightTarget = nightTarget ? 0 : 1; $('mc-night').textContent = nightTarget ? '☀️ Kun' : '🌙 Tun'; };
$('mc-rotate').onclick = () => { controls.autoRotate = !controls.autoRotate; $('mc-rotate').textContent = controls.autoRotate ? "⏸ Aylanishni to'xtatish" : '▶️ Aylantirish'; };
$('mc-speed').onclick = () => { speedMul = speedMul === 1 ? 2 : speedMul === 2 ? 0.5 : 1; $('mc-speed').textContent = `⏩ Tezlik: ${speedMul}x`; };
function msg(t, ms){ const m = $('mc-msg'); m.textContent = t; m.style.display = 'block'; if (ms) setTimeout(() => m.style.display = 'none', ms); }
$('mc-rec').onclick = () => {
  if (!window.MediaRecorder){ msg("Brauzeringiz video yozishni qo'llab-quvvatlamaydi", 4000); return; }
  const stream = renderer.domElement.captureStream(30);
  const type = MediaRecorder.isTypeSupported('video/webm;codecs=vp9') ? 'video/webm;codecs=vp9' : 'video/webm';
  const rec = new MediaRecorder(stream, {mimeType:type, videoBitsPerSecond:6e6});
  const chunks = [];
  rec.ondataavailable = e => e.data.size && chunks.push(e.data);
  rec.onstop = () => {
    const url = URL.createObjectURL(new Blob(chunks, {type:'video/webm'}));
    const a = document.createElement('a'); a.href = url; a.download = 'mini_shaharcha.webm';
    document.body.appendChild(a); a.click(); a.remove();
    msg('✅ Video tayyor: mini_shaharcha.webm yuklab olindi', 6000);
    $('mc-rec').disabled = false;
  };
  rec.start(); $('mc-rec').disabled = true;
  let left = 10; msg(`🔴 Yozilmoqda... ${left}`);
  const iv = setInterval(() => { left--; if (left > 0) msg(`🔴 Yozilmoqda... ${left}`); else { clearInterval(iv); rec.stop(); } }, 1000);
};

// ---------- Animatsiya sikli ----------
const clock = new THREE.Clock();
let t = 0;
function place(obj, p, flip){
  obj.position.x = p.x; obj.position.z = p.z;
  const dx = flip ? -p.dx : p.dx, dz = flip ? -p.dz : p.dz;
  obj.rotation.y = Math.atan2(-dz, dx);
}
function animate(){
  requestAnimationFrame(animate);
  const dt = Math.min(clock.getDelta(), 0.05) * speedMul; t += dt;

  night += (nightTarget - night) * Math.min(1, dt*2/speedMul);
  applyNight(night);

  for (const c of cars){
    c.s += c.speed * dt;
    place(c.g, c.loop.at(c.s), c.speed < 0);
    c.g.position.y = Y0;
    c.wheels.forEach(w => w.rotation.y += Math.abs(c.speed) * dt / 0.06);
  }
  for (const p of people){
    p.s += p.speed * dt;
    place(p.g, p.loop.at(p.s), p.speed < 0);
    const ph = t*9 + p.phase;
    p.g.position.y = Y0 + Math.abs(Math.sin(ph))*0.012;
    p.legs[0].rotation.z = Math.sin(ph)*0.5; p.legs[1].rotation.z = -Math.sin(ph)*0.5;
    p.arms[0].rotation.z = -Math.sin(ph)*0.4; p.arms[1].rotation.z = Math.sin(ph)*0.4;
  }
  kids.forEach((k, i) => {
    k.angle += dt*0.6;
    const r = 0.68;
    k.g.position.set(fx + Math.cos(k.angle)*r, Y0, fz + Math.sin(k.angle)*r);
    k.g.rotation.y = Math.atan2(-Math.cos(k.angle), -Math.sin(k.angle));
    const ph = t*10 + i;
    k.legs[0].rotation.z = Math.sin(ph)*0.6; k.legs[1].rotation.z = -Math.sin(ph)*0.6;
    k.arms[0].rotation.z = -Math.sin(ph)*0.5; k.arms[1].rotation.z = Math.sin(ph)*0.5;
  });

  wheel.rotation.z += dt*0.35;
  cabins.forEach((c, i) => {   // kabinalar doim pastga osilib turadi
    const a = i/10*Math.PI*2 + wheel.rotation.z;
    c.position.set(Math.cos(a)*0.85, Math.sin(a)*0.85 - 0.1, 0);
  });

  drops.forEach(d => {
    const u = d.userData; u.t += dt*1.4; if (u.t > 1){ u.t = 0; u.a = Math.random()*Math.PI*2; u.v = rand(.18, .3); }
    const r = u.v * u.t * 1.1;
    d.position.set(fx + Math.cos(u.a)*r, Y0 + 0.37 + 0.45*u.t - 0.55*u.t*u.t, fz + Math.sin(u.a)*r);
  });
  water.position.y = Y0 + 0.12 + Math.sin(t*3)*0.003;

  trees.forEach(g => { g.rotation.z = Math.sin(t*1.5 + g.userData.phase)*0.03; });

  steam.forEach(s => {
    s.userData.t = (s.userData.t + dt*0.25) % 1;
    const u = s.userData.t;
    s.position.set(-1.25 - u*0.3 + Math.sin(u*9 + t)*0.08, 1.35 + u*1.6, Math.cos(u*7)*0.08);
    s.scale.setScalar(0.6 + u*1.8);
    s.material.opacity = 0.35 * Math.sin(u*Math.PI);
  });

  const ha = t*0.35;
  heli.position.set(city.position.x + Math.cos(ha)*4.2, TABLE_Y + 5.2 + Math.sin(t*1.3)*0.25, Math.sin(ha)*3.2);
  heli.rotation.y = Math.atan2(-Math.cos(ha), -Math.sin(ha));
  heli.rotation.z = 0.12;
  rotor.rotation.y += dt*40;

  controls.update();
  renderer.render(scene, camera);
}
window.__miniCity = {camera, controls};
animate();

new ResizeObserver(() => { renderer.setSize(W(), H()); camera.aspect = W()/H(); camera.updateProjectionMatrix(); }).observe(view);
})();
</script>
'''

display(HTML(html))

## 📱 Telefonda ko'rish uchun faylni yuklab olish (ixtiyoriy)

Quyidagi katakni ishga tushirsangiz, animatsiya `mini_shaharcha.html` fayli sifatida kompyuteringizga yuklanadi. Bu faylni istalgan brauzerda (telefonda ham) ochsangiz, internet bo'lsa animatsiya ishlaydi.

In [ ]:
page = '<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">' \
       '<title>Mini shaharcha</title></head><body style="margin:0;background:#dfe9f3">' + \
       html.replace('height:620px', 'height:100vh').replace('border-radius:12px', 'border-radius:0') + '</body></html>'
with open('mini_shaharcha.html', 'w', encoding='utf-8') as f:
    f.write(page)

try:
    from google.colab import files
    files.download('mini_shaharcha.html')
except ImportError:
    print("Fayl saqlandi: mini_shaharcha.html")